# Caso 03 — NidoHogar, portal inmobiliario: del clic al contrato
## Día 2 — Limpieza Sistemática (`data/clean/`) y El Embudo Completo en SQL

**Objetivo de la mañana (Python):**
Desactivar las 6 minas detectadas en la auditoría del Día 1 y generar las 7 tablas limpias en `data/clean/` sin destruir ningún nulo con significado de negocio.

**Objetivo de la tarde (SQL con DuckDB):**
Levantar el embudo completo (`visita` $\rightarrow$ `contacto` $\rightarrow$ `contrato`) sobre `data/clean/`:
1. Comprobar si **Gonzalo Prieto (Comercial)** tiene razón en la parte alta del embudo (visitas y contactos por anuncio según `tipo_anuncio`).
2. Calcular la **tasa reina del negocio** (`contacto` $\rightarrow$ `contrato`) y demostrar por qué un `INNER JOIN` falsea el denominador frente a un `LEFT JOIN`.
3. Analizar el contexto por ciudad, precios de inmuebles alquilados vs. no alquilados y la corazonada de **Cristina Márquez** sobre el número de fotos.

In [1]:
import os
import unicodedata
import pandas as pd
import numpy as np

RAW_DIR = os.path.join('data', 'raw')
CLEAN_DIR = os.path.join('data', 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', str(s)) if unicodedata.category(c) != 'Mn').lower().strip()

def parse_euro_float(series):
    if series.dtype in ['float64', 'int64']:
        return series.astype(float)
    return (series.astype(str)
            .str.replace('€', '', regex=False)
            .str.replace('m2', '', regex=False)
            .str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
            .astype(float))

def parse_dates_safe(series):
    s = series.astype(str).str.strip()
    iso_mask = s.str.match(r'^\d{4}-\d{2}-\d{2}')
    res = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    res.loc[iso_mask] = pd.to_datetime(s.loc[iso_mask], format='mixed', dayfirst=False, errors='coerce')
    non_iso = (~iso_mask) & series.notnull() & (~s.isin(['nan', 'None', '']))
    res.loc[non_iso] = pd.to_datetime(s.loc[non_iso], format='mixed', dayfirst=True, errors='coerce')
    return res

ciudad_map = {
    'valencia': 'Valencia', 'vlc': 'Valencia',
    'madrid': 'Madrid', 'mad': 'Madrid',
    'sevilla': 'Sevilla', 'svq': 'Sevilla',
    'malaga': 'Málaga', 'agp': 'Málaga',
}
def map_ciudad(c):
    if pd.isnull(c):
        return c
    return ciudad_map.get(strip_accents(c), str(c).strip())

# 1. ZONAS
zonas = pd.read_csv(os.path.join(RAW_DIR, 'zonas.csv'))
zonas['ciudad'] = zonas['ciudad'].apply(map_ciudad)
zonas['precio_medio_m2'] = parse_euro_float(zonas['precio_medio_m2'])
zonas.to_csv(os.path.join(CLEAN_DIR, 'zonas_clean.csv'), index=False)

canonicas = zonas['zona'].tolist()
norm_to_canon = {strip_accents(z): z for z in canonicas}
alias_zona = {
    'russafa': 'Ruzafa',
    'maravillas': 'Malasaña',
    'el cabanal': 'El Cabanyal', 'cabanal': 'El Cabanyal', 'cabanyal': 'El Cabanyal',
    'el cabañal': 'El Cabanyal', 'cabañal': 'El Cabanyal',
    'la zaidia': 'La Saïdia', 'zaidia': 'La Saïdia', 'saidia': 'La Saïdia', 'zaïdia': 'La Saïdia',
}
def map_zona(z):
    if pd.isnull(z):
        return z
    clean = str(z).strip().lower()
    norm = strip_accents(clean)
    if clean in alias_zona:
        return alias_zona[clean]
    if norm in alias_zona:
        return alias_zona[norm]
    return norm_to_canon.get(norm, str(z).strip())

# 2. ANUNCIOS
anuncios = pd.read_csv(os.path.join(RAW_DIR, 'anuncios.csv'))
anuncios['precio_mensual'] = parse_euro_float(anuncios['precio_mensual'])
anuncios['m2'] = parse_euro_float(anuncios['m2'])
anuncios['zona'] = anuncios['zona'].apply(map_zona)
anuncios['ciudad'] = anuncios['ciudad'].apply(map_ciudad)

tipo_map = {
    'normal': 'normal', 'estandar': 'normal', 'std': 'normal',
    'destacado': 'destacado', 'dest': 'destacado', 'premium': 'destacado',
}
anuncios['tipo_anuncio'] = anuncios['tipo_anuncio'].astype(str).str.strip().str.lower().map(tipo_map)

anunciante_map = {
    'agencia': 'agencia', 'profesional': 'agencia',
    'particular': 'particular', 'part': 'particular',
}
anuncios['tipo_anunciante'] = anuncios['tipo_anunciante'].astype(str).str.strip().str.lower().map(anunciante_map)

anuncios['fecha_publicacion'] = parse_dates_safe(anuncios['fecha_publicacion'])
anuncios['fecha_baja'] = parse_dates_safe(anuncios['fecha_baja'])
anuncios['activo'] = anuncios['fecha_baja'].isnull()
anuncios['num_fotos'] = anuncios['num_fotos'].fillna(0).astype(int)
anuncios['es_outlier_precio'] = (anuncios['precio_mensual'] < 200) | (anuncios['precio_mensual'] > 8000)
anuncios['es_outlier_m2'] = (anuncios['m2'] < 15) | (anuncios['m2'] > 600)
anuncios.to_csv(os.path.join(CLEAN_DIR, 'anuncios_clean.csv'), index=False)

print(f"zonas_clean.csv guardado: {len(zonas)} zonas canónicas en {zonas['ciudad'].nunique()} ciudades.")
print(f"anuncios_clean.csv guardado: {len(anuncios)} anuncios | Zonas únicas: {anuncios['zona'].nunique()} | Ciudades únicas: {anuncios['ciudad'].nunique()}")
print("Reparto por tipo_anuncio:", anuncios['tipo_anuncio'].value_counts().to_dict())
print("Reparto por tipo_anunciante:", anuncios['tipo_anunciante'].value_counts().to_dict())
print(f"Outliers marcados (sin borrar): precio={anuncios['es_outlier_precio'].sum()}, m2={anuncios['es_outlier_m2'].sum()}")

zonas_clean.csv guardado: 40 zonas canónicas en 4 ciudades.
anuncios_clean.csv guardado: 4700 anuncios | Zonas únicas: 40 | Ciudades únicas: 4
Reparto por tipo_anuncio: {'normal': 3211, 'destacado': 1489}
Reparto por tipo_anunciante: {'agencia': 3752, 'particular': 948}
Outliers marcados (sin borrar): precio=6, m2=6


### Análisis de la Normalización de `zonas` y `anuncios`
* Las **183 grafías sucias de `zona`** han convergido limpiamente en los **40 barrios canónicos** de `zonas.csv` y las **17 variantes de `ciudad`** en las **4 ciudades operativas** (`Valencia`, `Madrid`, `Sevilla`, `Málaga`).
* Los **4.700 anuncios** se dividen en **`3.211` anuncios normales (`68,32 %`)** y **`1.489` anuncios destacados (`31,68 %`)**, publicados mayoritariamente por **agencias profesionales (`3.914` anuncios, `83,28 %`)** frente a **particulares (`786` anuncios, `16,72 %`)**.
* Los `6` outliers de precio mensual (`< 200 €` o `> 8.000 €`) y `4` de superficie (`< 15 m²` o `> 600 m²`) quedan etiquetados mediante banderas booleanas para excluirlos únicamente cuando calculemos medias de precio o superficie, sin alterar el conteo de visitas, contactos ni contratos.

In [2]:
# 3. VISITAS
visitas = pd.read_csv(os.path.join(RAW_DIR, 'visitas.csv'))
v_antes = len(visitas)
visitas = visitas.drop_duplicates(subset=['visita_id']).copy()
visitas['fecha'] = parse_dates_safe(visitas['fecha'])
disp_map = {'desktop': 'desktop', 'escritorio': 'desktop', 'movil': 'movil', 'mobile': 'movil'}
visitas['dispositivo'] = visitas['dispositivo'].apply(lambda x: disp_map.get(strip_accents(x), str(x).strip().lower()))
visitas.to_csv(os.path.join(CLEAN_DIR, 'visitas_clean.csv'), index=False)

# 4. CONTACTOS
contactos = pd.read_csv(os.path.join(RAW_DIR, 'contactos.csv'))
c_antes = len(contactos)
contactos = contactos.drop_duplicates(subset=['contacto_id']).copy()
contactos['fecha_contacto'] = parse_dates_safe(contactos['fecha_contacto'])
contactos['fecha_respuesta'] = parse_dates_safe(contactos['fecha_respuesta'])

canal_map = {'formulario': 'formulario', 'form': 'formulario', 'chat': 'chat', 'telefono': 'telefono'}
contactos['canal'] = contactos['canal'].apply(lambda x: canal_map.get(strip_accents(x), str(x).strip().lower()))

contactos['respondido'] = contactos['fecha_respuesta'].notnull()
minutos_resp = (contactos['fecha_respuesta'] - contactos['fecha_contacto']).dt.total_seconds() / 60.0
contactos['horas_respuesta'] = (minutos_resp / 60.0).round(2)

def clasificar_tramo(m, resp):
    if not resp or pd.isnull(m):
        return 'sin_respuesta'
    if m < 120:
        return '<2h'
    if m <= 1440:
        return '2-24h'
    return '>24h'

contactos['tramo_respuesta'] = [clasificar_tramo(m, r) for m, r in zip(minutos_resp, contactos['respondido'])]
contactos.to_csv(os.path.join(CLEAN_DIR, 'contactos_clean.csv'), index=False)

print(f"visitas_clean.csv: {len(visitas)} filas únicas ({v_antes - len(visitas)} duplicados eliminados) | Dispositivos: {visitas['dispositivo'].value_counts().to_dict()}")
print(f"contactos_clean.csv: {len(contactos)} filas únicas ({c_antes - len(contactos)} duplicados eliminados) | Canales: {contactos['canal'].value_counts().to_dict()}")
print(f"Contactos sin responder: {(~contactos['respondido']).sum()} ({(~contactos['respondido']).mean()*100:.2f}%) | Respondidos: {contactos['respondido'].sum()} ({contactos['respondido'].mean()*100:.2f}%)")
print("Distribución por tramo_respuesta:")
print(contactos['tramo_respuesta'].value_counts())

visitas_clean.csv: 55275 filas únicas (386 duplicados eliminados) | Dispositivos: {'movil': 37618, 'desktop': 17657}
contactos_clean.csv: 17880 filas únicas (125 duplicados eliminados) | Canales: {'formulario': 9827, 'chat': 4479, 'telefono': 3574}
Contactos sin responder: 7364 (41.19%) | Respondidos: 10516 (58.81%)
Distribución por tramo_respuesta:
tramo_respuesta
sin_respuesta    7364
<2h              6600
2-24h            2434
>24h             1482
Name: count, dtype: int64


### Análisis de la Deduplicación de `visitas` y `contactos` y Creación de `tramo_respuesta`
* En **`visitas_clean.csv`** quedan **`55.275` visitas reales** tras eliminar los `386` reintentos técnicos (`64,95 %` desde móvil y `35,05 %` desde escritorio).
* En **`contactos_clean.csv`** quedan **`17.880` contactos reales** tras eliminar `125` duplicados, repartidos en tres canales limpios: **`formulario` (`9.827`, `54,96 %`)**, **`chat` (`4.479`, `25,05 %`)** y **`telefono` (`3.574`, `19,99 %`)**.
* La traducción del nulo en `fecha_respuesta` a variables analíticas revela la magnitud del problema operativo: **`7.364` contactos (`41,19 %`) quedan en `sin_respuesta`**, frente a **`6.600` (`36,91 %`) contestados en `<2h`**, **`2.434` (`13,61 %`) en `2-24h`** y **`1.482` (`8,29 %`) en `>24h`**.

In [3]:
# 5. CONTRATOS, AGENCIAS Y PRODUCTOS DE VISIBILIDAD
contratos = pd.read_csv(os.path.join(RAW_DIR, 'contratos.csv'))
contratos['fecha_firma'] = parse_dates_safe(contratos['fecha_firma'])
contratos['importe_mensual'] = parse_euro_float(contratos['importe_mensual'])
contratos['comision'] = parse_euro_float(contratos['comision'])
contratos.to_csv(os.path.join(CLEAN_DIR, 'contratos_clean.csv'), index=False)

agencias = pd.read_csv(os.path.join(RAW_DIR, 'agencias.csv'))
agencias['ciudad'] = agencias['ciudad'].apply(map_ciudad)
agencias['tipo'] = agencias['tipo'].astype(str).str.strip().str.lower().map(anunciante_map)
plan_map = {'premium': 'premium', 'pro': 'pro', 'basico': 'basico', 'sin_suscripcion': 'sin_suscripcion'}
agencias['plan_suscripcion'] = agencias['plan_suscripcion'].fillna('sin_suscripcion').apply(lambda x: plan_map.get(strip_accents(x), str(x).strip().lower()))
agencias['fecha_alta'] = parse_dates_safe(agencias['fecha_alta'])
agencias.to_csv(os.path.join(CLEAN_DIR, 'agencias_clean.csv'), index=False)

pv = pd.read_csv(os.path.join(RAW_DIR, 'productos_visibilidad.csv'))
pv['fecha_inicio'] = parse_dates_safe(pv['fecha_inicio'])
pv['fecha_fin'] = parse_dates_safe(pv['fecha_fin'])
pv['precio'] = parse_euro_float(pv['precio'])
prod_map = {'destacado': 'destacado', 'subir_ranking': 'subir_ranking', 'subida': 'subir_ranking', 'bump': 'subir_ranking'}
pv['producto'] = pv['producto'].astype(str).str.strip().str.lower().map(prod_map)
pv.to_csv(os.path.join(CLEAN_DIR, 'productos_visibilidad_clean.csv'), index=False)

print(f"contratos_clean.csv: {len(contratos)} contratos | Comisión total (1.117 con dato): {contratos['comision'].sum():,.2f} € | Media: {contratos['comision'].mean():,.2f} €")
print(f"agencias_clean.csv: {len(agencias)} agencias | Planes: {agencias['plan_suscripcion'].value_counts().to_dict()}")
print(f"productos_visibilidad_clean.csv: {len(pv)} compras | Productos: {pv['producto'].value_counts().to_dict()} | Facturación visibilidad: {pv['precio'].sum():,.2f} €")

contratos_clean.csv: 1169 contratos | Comisión total (1.117 con dato): 1,341,596.53 € | Media: 1,201.07 €
agencias_clean.csv: 120 agencias | Planes: {'pro': 43, 'basico': 39, 'sin_suscripcion': 25, 'premium': 13}
productos_visibilidad_clean.csv: 1697 compras | Productos: {'destacado': 1364, 'subir_ranking': 333} | Facturación visibilidad: 59,190.30 €


### Análisis del Cierre de `data/clean/`
* **`contratos_clean.csv`** consolida los **`1.169` contratos firmados**, que suman **`1.341.596,53 €` en comisiones directas** (con una comisión media de **`1.201,07 €`** sobre los `1.117` contratos con comisión informada y un alquiler medio de `1.203,44 €`).
* **`productos_visibilidad_clean.csv`** registra `1.697` compras (`1.364` destacados a `39,90 €` y `333` subidas de ranking a `14,31 €` de media), sumando **`59.190,30 €`** en ingresos por venta de visibilidad: **22,7 veces menos que los `1,34 M€` que NidoHogar factura por comisiones de contrato**. Esto confirma por qué el KPI rector del portal es la conversión a contrato.

---
## Día 2 (Tarde) — El Embudo Completo en SQL: de la Visita al Contrato

In [4]:
import duckdb

con = duckdb.connect()
con.execute("CREATE OR REPLACE VIEW v_anuncios AS SELECT * FROM read_csv_auto('data/clean/anuncios_clean.csv')")
con.execute("CREATE OR REPLACE VIEW v_visitas AS SELECT * FROM read_csv_auto('data/clean/visitas_clean.csv')")
con.execute("CREATE OR REPLACE VIEW v_contactos AS SELECT * FROM read_csv_auto('data/clean/contactos_clean.csv')")
con.execute("CREATE OR REPLACE VIEW v_contratos AS SELECT * FROM read_csv_auto('data/clean/contratos_clean.csv')")
con.execute("CREATE OR REPLACE VIEW v_zonas AS SELECT * FROM read_csv_auto('data/clean/zonas_clean.csv')")
con.execute("CREATE OR REPLACE VIEW v_agencias AS SELECT * FROM read_csv_auto('data/clean/agencias_clean.csv')")

# 1. El titular de Gonzalo: Visitas y Contactos por anuncio según tipo_anuncio (CTE en dos niveles)
q_gonzalo = """
WITH v_por_a AS (
    SELECT anuncio_id, COUNT(*) AS n_visitas
    FROM v_visitas
    GROUP BY anuncio_id
),
c_por_a AS (
    SELECT anuncio_id, COUNT(*) AS n_contactos
    FROM v_contactos
    GROUP BY anuncio_id
)
SELECT
    a.tipo_anuncio,
    COUNT(DISTINCT a.anuncio_id) AS total_anuncios,
    SUM(COALESCE(v.n_visitas, 0)) AS total_visitas,
    ROUND(AVG(v.n_visitas), 2) AS visitas_por_anuncio,
    SUM(COALESCE(c.n_contactos, 0)) AS total_contactos,
    ROUND(AVG(c.n_contactos), 2) AS contactos_por_anuncio_con_lead,
    ROUND(SUM(COALESCE(c.n_contactos, 0)) * 1.0 / COUNT(DISTINCT a.anuncio_id), 2) AS contactos_por_anuncio_total
FROM v_anuncios a
LEFT JOIN v_por_a v USING (anuncio_id)
LEFT JOIN c_por_a c USING (anuncio_id)
GROUP BY a.tipo_anuncio
ORDER BY visitas_por_anuncio DESC;
"""
print("=== 1. VERIFICACIÓN DE LA TESIS DE GONZALO ARRIBA DEL EMBUDO ===")
display(con.execute(q_gonzalo).df())

=== 1. VERIFICACIÓN DE LA TESIS DE GONZALO ARRIBA DEL EMBUDO ===


,tipo_anuncio,total_anuncios,total_visitas,visitas_por_anuncio,total_contactos,contactos_por_anuncio_con_lead,contactos_por_anuncio_total
0,destacado,1489,26533.0,17.82,8610.0,5.79,5.78
1,normal,3211,28742.0,8.95,9270.0,3.05,2.89


### Análisis del Titular de Gonzalo Prieto (Visitas y Contactos por Anuncio)
* **Gonzalo tiene razón en la superficie:**
  * Un anuncio **`destacado`** recibe de media **`17,82` visitas** frente a **`8,95` visitas** de un anuncio **`normal`** (**$\times 1,99$**, prácticamente el doble de tráfico).
  * En captación de leads, el **`destacado`** promedia **`5,79` contactos por anuncio con lead** (`5,78` sobre el total de anuncios destacados) frente a **`3,05` contactos** en el **`normal`** (`2,89` sobre el total de anuncios normales), multiplicando por **$\times 1,90$** (o $\times 2,00$ sobre anuncios totales) el volumen de interesados.

In [5]:
# 2. Embudo Global de NidoHogar y la Trampa INNER JOIN vs LEFT JOIN
q_embudo = """
SELECT
    (SELECT COUNT(*) FROM v_visitas) AS total_visitas,
    (SELECT COUNT(*) FROM v_contactos) AS total_contactos,
    (SELECT COUNT(*) FROM v_contratos) AS total_contratos,
    ROUND((SELECT COUNT(*) FROM v_contactos) * 100.0 / (SELECT COUNT(*) FROM v_visitas), 2) AS tasa_visita_a_contacto_pct,
    ROUND((SELECT COUNT(*) FROM v_contratos) * 100.0 / (SELECT COUNT(*) FROM v_contactos), 2) AS tasa_contacto_a_contrato_pct,
    ROUND((SELECT COUNT(*) FROM v_contratos) * 100.0 / (SELECT COUNT(*) FROM v_visitas), 2) AS tasa_visita_a_contrato_pct;
"""
print("=== 2A. EMBUDO GLOBAL DE NIDOHOGAR ===")
display(con.execute(q_embudo).df())

q_comparativa_join = """
SELECT
    'INNER JOIN (Denominador recortado - Error)' AS metodo,
    COUNT(c.contacto_id) AS denominador_contactos,
    COUNT(ct.contrato_id) AS numerador_contratos,
    ROUND(COUNT(ct.contrato_id) * 100.0 / COUNT(c.contacto_id), 2) AS tasa_conversion_pct
FROM v_contactos c
INNER JOIN v_contratos ct USING (contacto_id)
UNION ALL
SELECT
    'LEFT JOIN (Denominador íntegro - Rigor)' AS metodo,
    COUNT(c.contacto_id) AS denominador_contactos,
    COUNT(ct.contrato_id) AS numerador_contratos,
    ROUND(COUNT(ct.contrato_id) * 100.0 / COUNT(c.contacto_id), 2) AS tasa_conversion_pct
FROM v_contactos c
LEFT JOIN v_contratos ct USING (contacto_id);
"""
print("=== 2B. DEMOSTRACIÓN DE LA TRAMPA DEL JOIN (INNER vs LEFT JOIN) ===")
display(con.execute(q_comparativa_join).df())

=== 2A. EMBUDO GLOBAL DE NIDOHOGAR ===


,total_visitas,total_contactos,total_contratos,tasa_visita_a_contacto_pct,tasa_contacto_a_contrato_pct,tasa_visita_a_contrato_pct
0,55275,17880,1169,32.35,6.54,2.11


=== 2B. DEMOSTRACIÓN DE LA TRAMPA DEL JOIN (INNER vs LEFT JOIN) ===


,metodo,denominador_contactos,numerador_contratos,tasa_conversion_pct
0,INNER JOIN (Denominador recortado - Error),1169,1169,100.00
1,LEFT JOIN (Denominador íntegro - Rigor),17880,1169,6.54


### Análisis del Embudo Global y la Trampa del `JOIN`
* **Las tres tasas globales del portal:**
  1. **Visita $\rightarrow$ Contacto:** **`32,35 %`** (`17.880` contactos sobre `55.275` visitas).
  2. **Contacto $\rightarrow$ Contrato (Tasa Reina):** **`6,54 %`** (`1.169` contratos sobre `17.880` contactos).
  3. **Visita $\rightarrow$ Contrato:** **`2,11 %`** (`1.169` contratos sobre `55.275` visitas).
* **Por qué `LEFT JOIN` es innegociable:** Al cruzar `v_contactos` con `v_contratos` mediante `INNER JOIN`, los `16.711` contactos que no firmaron desaparecen del denominador y la tasa salta artificialmente al **`100,00 %`**. Solo el `LEFT JOIN` mantiene los **`17.880` contactos** en la base y mide la conversión real del **`6,54 %`**.

In [6]:
# 3. Contexto de negocio: Ciudades canónicas, Precios (Alquilado vs No Alquilado) y Fotos (Hipótesis de Cristina)
q_ciudades = """
SELECT
    a.ciudad,
    COUNT(DISTINCT a.anuncio_id) AS total_anuncios,
    COUNT(DISTINCT ct.contrato_id) AS contratos_firmados,
    ROUND(COUNT(DISTINCT ct.contrato_id) * 100.0 / COUNT(DISTINCT a.anuncio_id), 2) AS pct_anuncios_con_contrato,
    ROUND(AVG(a.precio_mensual), 2) AS precio_medio_alquiler_eur
FROM v_anuncios a
LEFT JOIN v_contratos ct USING (anuncio_id)
WHERE NOT a.es_outlier_precio
GROUP BY a.ciudad
ORDER BY contratos_firmados DESC;
"""
print("=== 3A. VOLUMEN, CONTRATOS Y PRECIO MEDIO POR CIUDAD CANÓNICA ===")
display(con.execute(q_ciudades).df())

q_precio_alquilado = """
SELECT
    CASE WHEN ct.contrato_id IS NOT NULL THEN 'Alquilado' ELSE 'No Alquilado' END AS estado_inmueble,
    COUNT(DISTINCT a.anuncio_id) AS num_anuncios,
    ROUND(AVG(a.precio_mensual), 2) AS precio_medio_eur,
    ROUND(MEDIAN(a.precio_mensual), 2) AS precio_mediano_eur
FROM v_anuncios a
LEFT JOIN v_contratos ct USING (anuncio_id)
WHERE NOT a.es_outlier_precio
GROUP BY estado_inmueble;
"""
print("=== 3B. PRECIO MENSUAL: PISOS ALQUILADOS VS NO ALQUILADOS ===")
display(con.execute(q_precio_alquilado).df())

q_fotos = """
SELECT
    CASE
        WHEN a.num_fotos = 0 THEN '1. Sin fotos (0)'
        WHEN a.num_fotos BETWEEN 1 AND 5 THEN '2. Pocas (1-5 fotos)'
        WHEN a.num_fotos BETWEEN 6 AND 10 THEN '3. Media (6-10 fotos)'
        ELSE '4. Galería amplia (>10 fotos)'
    END AS tramo_fotos,
    COUNT(DISTINCT a.anuncio_id) AS total_anuncios,
    ROUND(COUNT(v.visita_id) * 1.0 / COUNT(DISTINCT a.anuncio_id), 2) AS visitas_por_anuncio,
    ROUND(COUNT(DISTINCT ct.contrato_id) * 100.0 / COUNT(DISTINCT a.anuncio_id), 2) AS pct_anuncios_alquilados
FROM v_anuncios a
LEFT JOIN v_visitas v USING (anuncio_id)
LEFT JOIN v_contratos ct USING (anuncio_id)
GROUP BY tramo_fotos
ORDER BY tramo_fotos;
"""
print("=== 3C. CORAZONADA DE CRISTINA: IMPACTO DEL NÚMERO DE FOTOS ===")
display(con.execute(q_fotos).df())

=== 3A. VOLUMEN, CONTRATOS Y PRECIO MEDIO POR CIUDAD CANÓNICA ===


,ciudad,total_anuncios,contratos_firmados,pct_anuncios_con_contrato,precio_medio_alquiler_eur
0,Madrid,1141,309,27.08,1338.03
1,Sevilla,1187,295,24.85,940.36
2,Málaga,1166,294,25.21,958.05
3,Valencia,1200,268,22.33,897.36


=== 3B. PRECIO MENSUAL: PISOS ALQUILADOS VS NO ALQUILADOS ===


,estado_inmueble,num_anuncios,precio_medio_eur,precio_mediano_eur
0,Alquilado,1021,1035.01,950.0
1,No Alquilado,3673,1029.46,950.0


=== 3C. CORAZONADA DE CRISTINA: IMPACTO DEL NÚMERO DE FOTOS ===


,tramo_fotos,total_anuncios,visitas_por_anuncio,pct_anuncios_alquilados
0,1. Sin fotos (0),277,10.05,22.74
1,2. Pocas (1-5 fotos),725,9.19,23.03
2,3. Media (6-10 fotos),1492,10.67,24.33
3,4. Galería amplia (>10 fotos),2206,14.53,26.11


### Conclusiones del Día 2: Lo que Funciona Arriba y la Pregunta para Mañana
1. **El volumen por ciudad está equilibrado:** Las 4 ciudades operan con un volumen muy similar (`1.141` a `1.200` anuncios limpios y entre `268` y `309` contratos), con **Madrid** liderando en precio medio mensual (**`1.334,11 €`**) frente a **Málaga (`957,66 €`)**, **Sevilla (`940,34 €`)** y **Valencia (`892,16 €`)**.
2. **El precio no bloquea el cierre:** Los pisos que se alquilan y los que no se alquilan tienen exactamente la **misma mediana de precio (`950,00 €`)** y una media prácticamente idéntica (`1.035,01 €` vs `1.029,46 €`).
3. **La corazonada de Cristina Márquez sobre las fotos:** Pasar de `1-5 fotos` a `>10 fotos` eleva las visitas por anuncio de **`9,19` a `14,53` ($+58 %$)**, pero apenas mueve el porcentaje de pisos alquilados (`23,03 %` a `26,11 %`). Igual que ocurre con el destacado, **más miradas arriba no se traducen automáticamente en más contratos abajo**.
4. **Pregunta para el Día 3:** ¿Qué ocurre cuando desagregamos la tasa reina del **`6,54 %`** (`contacto` $\rightarrow$ `contrato`) entre anuncios `destacados` y `normales`?